# EcoHome Energy Advisor - Agent Run & Evaluation

Evaluation criteria: Accuracy, Relevance, Completeness, Usefulness, Tool appropriateness, Tool completeness.


## 1. Import and Initialize


In [1]:
import os, sys
from pathlib import Path

ROOT = Path("/Users/sandipdey2/Downloads/udacityprojects/langchain/langgraphenergy/ecohome_solution")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("cwd:", Path.cwd())
print("models exists:", (ROOT / "models" / "energy.py").exists())

cwd: /Users/sandipdey2/Downloads/udacityprojects/langchain/langgraphenergy/ecohome_solution
models exists: True


In [17]:
import os
from importlib import reload

os.environ["ECOHOME_DB_PATH"] = "/tmp/energy_data.db"

import tools, agent
reload(tools)
reload(agent)

print(tools.query_energy_usage.invoke({
    "start_date": "2026-08-25",
    "end_date": "2026-09-01",
    "device_type": "EV and HVAC",
})["by_device"])

from agent import Agent
ecohome_agent = Agent(instructions=ECOHOME_SYSTEM_PROMPT, model="gpt-4o-mini")

{'EV': {'kwh': 9460.5, 'cost_usd': 1076.21, 'records': 1008}, 'HVAC': {'kwh': 2000.72, 'cost_usd': 238.34, 'records': 1008}}


In [18]:
from datetime import datetime
from agent import Agent


In [3]:
ECOHOME_SYSTEM_PROMPT = """
Who you are
You are the EcoHome Energy Advisor, a smart-home energy optimization agent.
Your role is to help a household with rooftop solar, an EV, HVAC and appliances
cut electricity cost and environmental impact.

What you should do (steps)
1. Read the user question and any Location context.
2. Decide which tools you need. Never guess weather, prices or meter readings.
3. Call tools:
   - Scheduling / "when should I" -> get_weather_forecast AND get_electricity_prices
   - "my usage / history" -> get_recent_energy_summary or query_energy_usage
   - "how much can I save" -> get_electricity_prices AND calculate_energy_savings
   - Behaviour advice -> search_energy_tips
4. Analyse peak vs off-peak rates, solar-friendly hours and device totals.
5. Write the recommendation.

Key capabilities
- Weather integration and solar-window prediction
- Time-of-use price optimization
- Historical usage analysis
- RAG over the energy-saving knowledge base
- Multi-device plans (EV, HVAC, appliances, pool, battery)
- Savings / simple ROI arithmetic

Recommendation instructions
- Give concrete clock hours (e.g. 11:00-14:00)
- Include a dollar figure when a tariff shift is involved
- Explain why in one line that cites the tool data
- Cite a knowledge-base source filename
- End with one next step the customer can take today
- Never put EV charging on 16:00-21:00 peak unless the battery is critically low
- A dishwasher cycle is ~1.2-1.5 kWh. Never treat a 24-hour appliance total as one load.
- When reporting meter history, quote by_device kWh from query_energy_usage; do not say 0 if totals exist.

Example questions
- When should I charge my electric car tomorrow to minimize cost and maximize solar power?
- What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?
- Suggest three ways I can reduce energy use based on my usage history.
- How much can I save by running my dishwasher during off-peak hours?
- What's the best time to run my pool pump this week based on the weather forecast?
"""




In [19]:
ecohome_agent = Agent(instructions=ECOHOME_SYSTEM_PROMPT, model="gpt-4o-mini")


In [20]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA",
)
print(response["messages"][-1].content)
print("TOOLS:")
for msg in response["messages"]:
    obj = msg.model_dump() if hasattr(msg, "model_dump") else {}
    if obj.get("tool_call_id"):
        print("-", getattr(msg, "name", obj.get("name")))



To minimize costs and maximize solar power when charging your electric vehicle (EV) tomorrow in San Francisco, you should charge between **12:00 PM and 3:00 PM**. 

During this time, solar generation is expected to be at its peak, with solar irradiance reaching up to **871 W/m²** around 2:00 PM. Additionally, electricity prices are lower during this period compared to the peak hours of **4:00 PM to 8:00 PM**, where rates can go as high as **$0.3182 per kWh**.

By charging during the solar-friendly hours, you can take advantage of both the solar energy produced and the lower electricity rates, which will help reduce your overall energy costs.

**Next step:** Schedule your EV charging for tomorrow between 12:00 PM and 3:00 PM.
TOOLS:
- get_weather_forecast
- get_electricity_prices
- query_solar_generation


## 2. Define Test Cases


In [21]:
test_cases = [
    {"id": "ev_charging_1", "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
     "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
     "expected_response": "time recommendation cost analysis solar"},
    {"id": "ev_charging_2", "question": "Is it cheaper to charge my EV at noon or at midnight if I have rooftop solar?",
     "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
     "expected_response": "compare solar noon off-peak cost"},
    {"id": "thermostat_1", "question": "What temperature should I set my thermostat on Wednesday afternoon if electricity prices spike?",
     "expected_tools": ["get_electricity_prices", "get_weather_forecast"],
     "expected_response": "setpoint pre-cool peak"},
    {"id": "thermostat_2", "question": "How should I pre-cool the house before the evening peak?",
     "expected_tools": ["get_electricity_prices"],
     "expected_response": "pre-cool window peak"},
    {"id": "appliance_1", "question": "How much can I save by running my dishwasher during off-peak hours?",
     "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
     "expected_response": "savings dollars off-peak"},
    {"id": "appliance_2", "question": "When should I run the washing machine and dryer this week?",
     "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
     "expected_response": "schedule hours solar off-peak"},
    {"id": "solar_1", "question": "How can I maximize self-consumption of my solar generation tomorrow?",
     "expected_tools": ["get_weather_forecast", "query_solar_generation"],
     "expected_response": "solar window self-consumption"},
    {"id": "history_1", "question": "Suggest three ways I can reduce energy use based on my usage history.",
     "expected_tools": ["get_recent_energy_summary", "search_energy_tips"],
     "expected_response": "three actions usage kWh"},
    {"id": "pool_1", "question": "What's the best time to run my pool pump this week based on the weather forecast?",
     "expected_tools": ["get_weather_forecast"],
     "expected_response": "pool pump hours solar"},
    {
        "id": "usage_query_1",
        "question": "How many kWh did my EV and HVAC use over the last 7 days according to the meter history?",
        "expected_tools": ["query_energy_usage"],
        "expected_response": "kWh by device last week from query_energy_usage",
    },
    {"id": "storage_1", "question": "Should I charge a home battery from solar at noon and discharge at 7pm?",
     "expected_tools": ["get_electricity_prices", "search_energy_tips"],
     "expected_response": "battery dispatch peak solar"},
]
if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")



## 3. Run Agent Tests


In [22]:
CONTEXT = 'Location: San Francisco, CA'


In [23]:
print("=== Running Agent Tests ===")
test_results = []
for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    try:
        response = ecohome_agent.invoke(question=test_case["question"], context=CONTEXT, reset_history=True)
        test_results.append({
            "test_id": test_case["id"],
            "question": test_case["question"],
            "response": response,
            "expected_tools": test_case["expected_tools"],
            "expected_response": test_case["expected_response"],
            "timestamp": datetime.now().isoformat(),
        })
        print(response["messages"][-1].content[:320], "...")
    except Exception as e:
        print("Error:", e)
        test_results.append({
            "test_id": test_case["id"],
            "question": test_case["question"],
            "response": f"Error: {e}",
            "expected_tools": test_case["expected_tools"],
            "expected_response": test_case["expected_response"],
            "timestamp": datetime.now().isoformat(),
            "error": str(e),
        })
print(f"\nCompleted {len(test_results)} tests")



=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------
To minimize costs and maximize solar power when charging your electric vehicle (EV) tomorrow in San Francisco, you should charge between **12:00 PM and 3:00 PM**. 

During this time, solar generation is expected to be at its peak, with solar irradiance reaching up to **871 W/m²** around 2:00 PM, which will help offset  ...

Test 2: ev_charging_2
Question: Is it cheaper to charge my EV at noon or at midnight if I have rooftop solar?
--------------------------------------------------
To determine whether it's cheaper to charge your EV at noon or at midnight, let's analyze the electricity prices and solar generation potential.

1. **Electricity Prices**:
   - **Noon (12:00 PM)**: The rate is **$0.258 per kWh** (mid-peak).
   - **Midnight (12:00 AM)**: The rate is **$0.1091 per kWh** (off-pea

## 4. Evaluate Responses


In [24]:
# LLM-as-a-judge (not keyword overlap)
from dotenv import load_dotenv
load_dotenv(".env", override=True)

import json, os
from langchain_openai import ChatOpenAI


def _judge_llm():
    key = os.getenv("VOCAREUM_API_KEY") or os.getenv("OPENAI_API_KEY")
    if not key:
        raise RuntimeError("Set VOCAREUM_API_KEY or OPENAI_API_KEY for the LLM judge")
    kwargs = {"model": os.getenv("ECOHOME_LLM_MODEL", "gpt-4o-mini"), "temperature": 0, "api_key": key}
    base = os.getenv("OPENAI_BASE_URL")
    if os.getenv("VOCAREUM_API_KEY") or base:
        kwargs["base_url"] = base or "https://openai.vocareum.com/v1"
    return ChatOpenAI(**kwargs)


def _parse_json(text):
    raw = (text or "").strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.lower().startswith("json"):
            raw = raw[4:].strip()
    start, end = raw.find("{"), raw.rfind("}")
    return json.loads(raw[start:end + 1])


def _ask_judge(system, user):
    msg = _judge_llm().invoke([
        {"role": "system", "content": system},
        {"role": "user", "content": user},
    ])
    return _parse_json(getattr(msg, "content", "") or "")


def evaluate_response(question, final_response, expected_response):
    """Semantic LLM-as-a-judge for ACCURACY, RELEVANCE, COMPLETENESS, USEFULNESS.

    Each score is 0-1 from the judge model, not token overlap.
    Feedback is case-specific (what was wrong / missing).
    """
    system = (
        "You are an expert examiner for a smart-home energy advisor. "
        "Judge the AGENT ANSWER against the QUESTION and EXPECTED CONCEPTS. "
        "Do not count keywords. Score meaning. "
        "Return ONLY JSON with numeric fields accuracy, relevance, completeness, "
        "usefulness (each 0-1) and feedback (object with those four keys, each a "
        "2-4 sentence critique). "
        "ACCURACY: are the facts, hours, prices and advice correct given typical "
        "TOU + solar guidance? Penalize peak EV charging 16:00-21:00 and invented meter data. "
        "RELEVANCE: does it actually answer this question (not just repeat words)? "
        "COMPLETENESS: hours, a dollar or kWh figure when relevant, a why, a next step. "
        "USEFULNESS: is the action safe, specific and usable today?"
    )
    user = (
        f"QUESTION:\n{question}\n\n"
        f"EXPECTED CONCEPTS:\n{expected_response}\n\n"
        f"AGENT ANSWER:\n{final_response}\n"
    )
    data = _ask_judge(system, user)
    scores = {}
    for key in ("accuracy", "relevance", "completeness", "usefulness"):
        scores[key] = float(max(0.0, min(1.0, float(data.get(key, 0)))))
    scores["overall"] = round(sum(scores.values()) / 4, 3)
    fb = data.get("feedback") or {}
    if isinstance(fb, dict):
        scores["feedback"] = [f"{k}: {v}" for k, v in fb.items()]
    elif isinstance(fb, list):
        scores["feedback"] = [str(x) for x in fb]
    else:
        scores["feedback"] = [str(fb)]
    return scores


def _used_tools(messages_list):
    used = []
    for msg in messages_list or []:
        name = getattr(msg, "name", None)
        dump = msg.model_dump() if hasattr(msg, "model_dump") else {}
        if dump.get("tool_call_id") and name:
            used.append(name)
        calls = getattr(msg, "tool_calls", None) or dump.get("tool_calls") or []
        for call in calls:
            n = call.get("name") if isinstance(call, dict) else getattr(call, "name", None)
            if n:
                used.append(n)
    out, seen = [], set()
    for n in used:
        if n not in seen:
            seen.add(n)
            out.append(n)
    return out


def evaluate_tool_usage(messages_list, expected_tools):
    """LLM judge: appropriateness and completeness are independent scores."""
    used = _used_tools(messages_list)
    expected = list(expected_tools or [])
    system = (
        "You evaluate tool routing for an energy advisor. "
        "Return ONLY JSON with appropriateness, completeness (0-1 each), "
        "and feedback (object with keys appropriateness, completeness). "
        "APPROPRIATENESS: were the tools that actually ran a good fit for the "
        "question? Extra useful tools are fine. Do NOT copy completeness. "
        "COMPLETENESS: were ALL expected tools used? Missing expected tools "
        "must lower this score even if extras ran."
    )
    user = f"EXPECTED TOOLS: {expected}\nTOOLS USED: {used}\n"
    data = _ask_judge(system, user)
    appropriateness = float(max(0.0, min(1.0, float(data.get("appropriateness", 0)))))
    completeness = float(max(0.0, min(1.0, float(data.get("completeness", 0)))))
    fb = data.get("feedback") or {}
    if isinstance(fb, dict):
        feedback = [f"{k}: {v}" for k, v in fb.items()]
    elif isinstance(fb, list):
        feedback = [str(x) for x in fb]
    else:
        feedback = [str(fb)]
    return {
        "appropriateness": appropriateness,
        "completeness": completeness,
        "overall": round((appropriateness + completeness) / 2, 3),
        "used_tools": used,
        "expected_tools": expected,
        "missing_tools": sorted(set(expected) - set(used)),
        "extra_tools": sorted(set(used) - set(expected)),
        "feedback": feedback,
    }


def _final_text(response):
    if not isinstance(response, dict):
        return str(response)
    messages = response.get("messages") or []
    if not messages:
        return str(response)
    return getattr(messages[-1], "content", "") or str(messages[-1])


def generate_evaluation_report(test_results):
    """Recommendations come from an LLM reviewer, not if-threshold strings."""
    rows = []
    for result in test_results:
        print("judging", result.get("test_id"), flush=True)
        resp = result.get("response") or {}
        messages = resp.get("messages") if isinstance(resp, dict) else []
        final = _final_text(resp)
        r_eval = evaluate_response(result.get("question"), final, result.get("expected_response", ""))
        t_eval = evaluate_tool_usage(messages, result.get("expected_tools", []))
        rows.append({
            "id": result.get("test_id"),
            "question": result.get("question"),
            "response_metrics": r_eval,
            "tool_metrics": t_eval,
            "preview": final[:180].replace("\n", " "),
            "error": result.get("error"),
        })
    n = max(len(rows), 1)
    summary = {
        "n_tests": len(rows),
        "mean_response": round(sum(r["response_metrics"]["overall"] for r in rows) / n, 3),
        "mean_tool": round(sum(r["tool_metrics"]["overall"] for r in rows) / n, 3),
        "accuracy": round(sum(r["response_metrics"]["accuracy"] for r in rows) / n, 3),
        "relevance": round(sum(r["response_metrics"]["relevance"] for r in rows) / n, 3),
        "completeness": round(sum(r["response_metrics"]["completeness"] for r in rows) / n, 3),
        "usefulness": round(sum(r["response_metrics"]["usefulness"] for r in rows) / n, 3),
        "tool_appropriateness": round(sum(r["tool_metrics"]["appropriateness"] for r in rows) / n, 3),
        "tool_completeness": round(sum(r["tool_metrics"]["completeness"] for r in rows) / n, 3),
    }
    strengths, weaknesses = [], []
    for label, val in [
        ("accuracy", summary["accuracy"]),
        ("relevance", summary["relevance"]),
        ("completeness", summary["completeness"]),
        ("usefulness", summary["usefulness"]),
        ("tool appropriateness", summary["tool_appropriateness"]),
        ("tool completeness", summary["tool_completeness"]),
    ]:
        (strengths if val >= 0.7 else weaknesses).append(f"{label}={val:.2f}")
    rec_json = _ask_judge(
        "You review an energy-advisor agent. Return JSON "
        '{"recommendations": ["...", "..."]} with 3 to 5 specific fixes. '
        "Use the missing tools and judge feedback. No generic advice.",
        json.dumps({"summary": summary, "cases": [
            {"id": r["id"], "missing": r["tool_metrics"].get("missing_tools"),
             "feedback": r["response_metrics"].get("feedback")}
            for r in rows
        ]}, default=str),
    )
    recommendations = [str(x) for x in (rec_json.get("recommendations") or [])]
    return {
        "summary": summary,
        "strengths": strengths,
        "weaknesses": weaknesses,
        "recommendations": recommendations,
        "cases": rows,
    }


def display_evaluation_report(report):
    s = report["summary"]
    print("=" * 72)
    print("ECOHOME ENERGY ADVISOR — EVALUATION REPORT (LLM-as-judge)")
    print("=" * 72)
    print(f"Tests run              : {s['n_tests']}")
    print(f"Mean response score    : {s['mean_response']:.2f}")
    print(f"Mean tool-usage score  : {s['mean_tool']:.2f}")
    print(f"  accuracy={s['accuracy']:.2f}  relevance={s['relevance']:.2f}  "
          f"completeness={s['completeness']:.2f}  usefulness={s['usefulness']:.2f}")
    print(f"  tool_appropriateness={s['tool_appropriateness']:.2f}  "
          f"tool_completeness={s['tool_completeness']:.2f}")
    print("\nStrengths")
    for item in report["strengths"]:
        print("  +", item)
    print("\nWeaknesses")
    for item in report["weaknesses"] or ["none flagged"]:
        print("  -", item)
    print("\nRecommendations (LLM reviewer)")
    for item in report["recommendations"]:
        print("  *", item)
    print("\nPer-case detail")
    for row in report["cases"]:
        rm, tm = row["response_metrics"], row["tool_metrics"]
        print(f"\n[{row['id']}] response={rm['overall']:.2f} tools={tm['overall']:.2f}")
        print("   Q:", (row.get("question") or "")[:90])
        print("   tools used:", tm.get("used_tools"), " missing:", tm.get("missing_tools"))
        print("   appropriateness:", tm.get("appropriateness"), " completeness:", tm.get("completeness"))
        for line in (rm.get("feedback") or []) + (tm.get("feedback") or []):
            print("   ", line)
        print("   preview:", row["preview"][:140])
    return report



from evaluation_report import generate_evaluation_report as _build_report
from evaluation_report import display_evaluation_report as _show_report

# Bind the LLM judges defined above into the structured report builder.
report = _build_report(test_results, evaluate_response, evaluate_tool_usage)
_show_report(report)





EcoHome Energy Advisor — Evaluation Report

1. Overall scores
   Tests                 : 11
   Composite             : 0.87
   Mean response         : 0.90
   Mean tool usage       : 0.84
   accuracy=0.79  relevance=0.98  completeness=0.91  usefulness=0.91
   tool_appropriateness=1.00  tool_completeness=0.69

2. Strengths
   + accuracy=0.79
   + relevance=0.98
   + completeness=0.91
   + usefulness=0.91
   + tool appropriateness=1.00

3. Weaknesses
   - tool completeness=0.69

4. Recommendations for improvement (LLM, from this run)
   1. ev_charging_2: Incorporate solar generation benefits into the analysis to accurately compare costs at noon and midnight.
   2. thermostat_1: Adjust the peak hours mentioned to accurately reflect typical TOU rates and provide a clearer rationale for pre-cooling.
   3. pool_1: Ensure the dates provided for running the pool pump correspond to the current week and account for peak pricing hours.
   4. storage_1: Advise against discharging the battery durin

{'title': 'EcoHome Energy Advisor — Evaluation Report',
 'overall_scores': {'n_tests': 11,
  'mean_response': 0.898,
  'mean_tool_usage': 0.843,
  'accuracy': 0.791,
  'relevance': 0.982,
  'completeness': 0.914,
  'usefulness': 0.905,
  'tool_appropriateness': 1.0,
  'tool_completeness': 0.686,
  'composite': 0.871},
 'strengths': [{'metric': 'accuracy', 'score': 0.791},
  {'metric': 'relevance', 'score': 0.982},
  {'metric': 'completeness', 'score': 0.914},
  {'metric': 'usefulness', 'score': 0.905},
  {'metric': 'tool appropriateness', 'score': 1.0}],
 'weaknesses': [{'metric': 'tool completeness', 'score': 0.686}],
 'recommendations': ['ev_charging_2: Incorporate solar generation benefits into the analysis to accurately compare costs at noon and midnight.',
  'thermostat_1: Adjust the peak hours mentioned to accurately reflect typical TOU rates and provide a clearer rationale for pre-cooling.',
  'pool_1: Ensure the dates provided for running the pool pump correspond to the current

In [25]:
out = ecohome_agent.invoke(
    "How many kWh did my EV and HVAC use over the last 7 days according to the meter history? "
    "Use start_date=2026-08-25 and end_date=2026-09-01 and quote by_device.",
    context=CONTEXT,
    reset_history=True,
)
print(out["messages"][-1].content)

Over the last 7 days (from August 25, 2026, to September 1, 2026), your energy usage was as follows:

- **Electric Vehicle (EV)**: 9,460.5 kWh
- **HVAC**: 2,000.72 kWh

This data indicates significant energy consumption, particularly from the EV, which may be due to frequent charging or long trips. 

If you're looking to optimize your energy usage or reduce costs, consider scheduling EV charging during solar-friendly hours or off-peak times. 

A next step you can take today is to review your charging schedule and adjust it to align with solar generation or off-peak electricity rates.
